# 🗺️ Etheras World Generator — v12

Procedural one-shot adventure generator for the Etheras setting.

**Usage:**
1. Run cell 2 (engine + WorldContext)
2. Run cell 3 (markdown exporter)
3. Run cell 4 (workflow) — generates and saves your adventure

Drop the output `.md` file into Claude Desktop for full narrative expansion.

---
**World data:** `mnt/data/world_mod/` 
**Exports:** `mnt/data/exports/`

In [1]:
# !pip install pyyaml  # uncomment if needed

import re, random, math, json, os, glob, time
from copy import deepcopy
from datetime import datetime
from textwrap import dedent
from typing import Any, Dict, List, Optional
from functools import reduce
import yaml

# ── Constants ────────────────────────────────────────────────────────────────
PROCGEN_TABLES     = 'procgen.tables'
PROCGEN_GENERATORS = 'procgen.generators'
PROCGEN_WEIGHTS    = 'procgen.weights'
PROCGEN_DICE       = 'procgen.dice'

DICE_PAT  = re.compile(r"(?:(\d+)[dD](\d+))|([()\+\-\*/])|(\d+)")
CURLY_RE  = re.compile(r"\{\{\s*([a-zA-Z0-9_.\-]+)(\|[a-zA-Z]+)?\s*\}\}")

# ── Module-level globals (updated by WorldContext) ───────────────────────────
CURRENT_ROOT  = None
CURRENT_RNG   = None
_TABLE_CACHE  = {}

# ════════════════════════════════════════════════════════════════════════════
# UTILITIES
# ════════════════════════════════════════════════════════════════════════════

def make_rng(seed: Optional[int]) -> random.Random:
    rng = random.Random()
    rng.seed(seed)
    return rng

def deep_merge(a: dict, b: dict) -> dict:
    out = a.copy()
    for k, v in b.items():
        if k in out and isinstance(out[k], dict) and isinstance(v, dict):
            out[k] = deep_merge(out[k], v)
        else:
            out[k] = deepcopy(v)
    return out

def load_yaml_file(path: str) -> dict:
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f) or {}

def load_world_folder(folder: str) -> dict:
    files = sorted(glob.glob(os.path.join(folder, '*.yaml')))
    if not files:
        return {}
    return reduce(deep_merge, [load_yaml_file(p) for p in files], {})

# ════════════════════════════════════════════════════════════════════════════
# DICE ENGINE
# ════════════════════════════════════════════════════════════════════════════

def _roll_single(rng: random.Random, ndice: int, sides: int) -> int:
    return sum(rng.randint(1, sides) for _ in range(ndice))

def roll(expr: str, rng: Optional[random.Random] = None) -> int:
    if not expr:
        raise ValueError("Empty dice expression")
    if rng is None:
        rng = CURRENT_RNG
    tokens = []
    for m in DICE_PAT.finditer(str(expr)):
        if m.group(1) and m.group(2):
            tokens.append(str(_roll_single(rng, int(m.group(1)), int(m.group(2)))))
        elif m.group(3):
            tokens.append(m.group(3))
        elif m.group(4):
            tokens.append(m.group(4))
    safe = ''.join(tokens)
    try:
        return int(eval(safe, {"__builtins__": {}}, {}))
    except Exception as e:
        raise ValueError(f"Bad dice expr '{expr}' -> '{safe}': {e}")

def weighted_choice(weights: Dict[str, int], rng: random.Random) -> str:
    if not weights:
        raise ValueError("Empty weights dict")
    items = list(weights.keys())
    wts   = [max(int(w), 0) for w in weights.values()]
    if sum(wts) <= 0:
        raise ValueError("Weights sum to zero")
    return rng.choices(items, weights=wts, k=1)[0]

# ════════════════════════════════════════════════════════════════════════════
# DATA NAVIGATION
# ════════════════════════════════════════════════════════════════════════════

def deep_get(d: Dict[str, Any], path: str) -> Any:
    cur = d
    for part in path.split('.'):
        if part == '':
            continue
        if isinstance(cur, dict) and part in cur:
            cur = cur[part]
        else:
            raise KeyError(f"Path '{path}' not found (stuck at '{part}')")
    return cur

def interpolate(s: str, ctx: Dict[str, Any]) -> str:
    def repl(m):
        key  = m.group(1)
        filt = (m.group(2) or '').lstrip('|')
        val  = ctx
        for p in key.split('.'):
            if isinstance(val, dict) and p in val:
                val = val[p]
            else:
                return m.group(0)
        sval = str(val)
        if   filt == 'title': sval = sval.title()
        elif filt == 'upper': sval = sval.upper()
        elif filt == 'lower': sval = sval.lower()
        return sval
    return CURLY_RE.sub(repl, s)

def resolve_at(ref: str, root: Dict[str, Any], runtime: Dict[str, Any]) -> Any:
    if not ref.startswith('@'):
        return ref
    path = ref[1:]
    if path.startswith('weights.'):
        path = f'{PROCGEN_WEIGHTS}.{path[8:]}'
    elif path.startswith('dice.'):
        path = f'{PROCGEN_DICE}.{path[5:]}'
    if '{{' in path:
        path = interpolate(path, runtime)
    if '[' in path and path.endswith(']'):
        head, key = path.split('[', 1)
        return deep_get(root, head)[key[:-1]]
    return deep_get(root, path)

# ════════════════════════════════════════════════════════════════════════════
# EXPRESSION EVALUATOR
# ════════════════════════════════════════════════════════════════════════════

def case(key: Any, mapping: Dict[str, Any], default=None):
    return mapping.get(str(key), default)

def treasure_tier_by_cr(cr_val: Any) -> Any:
    if CURRENT_ROOT is None:
        raise RuntimeError("CURRENT_ROOT not set")
    try:
        cr = float(cr_val)
    except Exception:
        cr = 0.0
    table = deep_get(CURRENT_ROOT, f'{PROCGEN_DICE}.treasure_tier_by_cr')
    exact = range_m = plus_m = None
    for k, v in table.items():
        k_str = str(k).strip()
        try:
            if float(k_str) == cr:
                exact = v; break
        except ValueError:
            pass
        if '-' in k_str:
            try:
                a, b = k_str.split('-', 1)
                if float(a) <= cr <= float(b):
                    range_m = v
            except ValueError:
                pass
        if k_str.endswith('+'):
            try:
                if cr >= float(k_str[:-1]):
                    plus_m = v
            except ValueError:
                pass
    return exact or range_m or plus_m or next(iter(table.values()))

def lookup(path: str) -> Any:
    if CURRENT_ROOT is None:
        raise RuntimeError("CURRENT_ROOT not set")
    return deep_get(CURRENT_ROOT, path)

class SafeLocals(dict):
    RESERVED = {"treasure_tier_by_cr","lookup","roll","case","math",
                "max","min","abs","round","int","float","len","sum"}
    def __init__(self, base_ctx):
        super().__init__()
        self._ctx = base_ctx
    def __contains__(self, key):
        return dict.__contains__(self, key) or (key in self._ctx and key not in self.RESERVED)
    def __getitem__(self, key):
        if dict.__contains__(self, key):
            return dict.__getitem__(self, key)
        if key in self._ctx and key not in self.RESERVED:
            return self._ctx[key]
        raise KeyError(key)
    def get(self, key, default=None):
        try:
            return self[key]
        except KeyError:
            return default

def eval_expr(expr: str, ctx: Dict[str, Any]) -> Any:
    sl = SafeLocals(ctx)
    sl.update({
        "case": case, "math": math,
        "treasure_tier_by_cr": treasure_tier_by_cr,
        "lookup": lookup,
        "roll": lambda s: roll(s, CURRENT_RNG),
        "max": max, "min": min, "abs": abs, "round": round,
        "int": int, "float": float, "len": len, "sum": sum,
    })
    try:
        return eval(expr, {"__builtins__": {}}, sl)
    except Exception as e:
        raise ValueError(f"Bad expr '{expr}': {e}")

# ════════════════════════════════════════════════════════════════════════════
# TABLE ENGINE
# ════════════════════════════════════════════════════════════════════════════

def build_table_cache(root: Dict[str, Any]) -> None:
    global _TABLE_CACHE
    try:
        tables = deep_get(root, PROCGEN_TABLES)
        _TABLE_CACHE = {t['id']: t for t in tables if 'id' in t}
    except KeyError:
        _TABLE_CACHE = {}

def get_proc_table(root: Dict[str, Any], table_id: str) -> Dict[str, Any]:
    if not _TABLE_CACHE:
        build_table_cache(root)
    if table_id not in _TABLE_CACHE:
        raise KeyError(f"Table '{table_id}' not found")
    return _TABLE_CACHE[table_id]

def roll_table(table: Dict[str, Any], rng: random.Random) -> Any:
    entries = table.get('entries', [])
    if not entries:
        return None
    total = sum(int(e.get('weight', 1)) for e in entries)
    r = rng.uniform(0, total)
    acc = 0
    for e in entries:
        acc += int(e.get('weight', 1))
        if r <= acc:
            return e.get('result', e)
    return entries[-1].get('result', entries[-1])

def run_generator(root, gen_name, rng, base_ctx=None):
    gens  = deep_get(root, PROCGEN_GENERATORS)
    if gen_name not in gens:
        raise KeyError(f"Generator '{gen_name}' not found")
    steps = gens[gen_name].get('steps', [])
    ctx   = {} if base_ctx is None else deepcopy(base_ctx)
    ctx['now'] = datetime.now(datetime.timezone.utc).isoformat() if hasattr(datetime, 'timezone') else datetime.utcnow().isoformat()

    for step in steps:
        if 'choose' in step:
            for k, v in step['choose'].items():
                resolved = resolve_at(v, root, ctx) if isinstance(v, str) and v.startswith('@') else v
                ctx[k] = weighted_choice(resolved, rng) if isinstance(resolved, dict) else resolved

        elif 'roll' in step:
            for k, v in step['roll'].items():
                expr = resolve_at(v, root, ctx) if isinstance(v, str) and v.startswith('@') else v
                ctx[k] = roll(str(expr), rng)

        elif 'pick_n' in step:
            spec    = step['pick_n']
            out_key = spec.get('into', 'problems')
            src     = spec.get('from', [])
            n       = roll(str(spec.get('n', 1)), rng) if isinstance(spec.get('n', 1), str) else int(spec.get('n', 1))
            pool    = list(src)
            rng.shuffle(pool)
            ctx[out_key] = pool[:n]

        elif 'derive' in step:
            for k, expr in step['derive'].items():
                ctx[k] = eval_expr(str(expr), ctx)

        elif 'table' in step:
            table_id = step['table']
            t   = get_proc_table(root, table_id)
            res = roll_table(t, rng)
            ctx[table_id] = res
            alias = table_id.split('_')[0]
            if alias and alias not in ctx:
                ctx[alias] = res

        elif 'bind' in step:
            for k, expr in step['bind'].items():
                ctx[k] = eval_expr(str(expr), ctx) if isinstance(expr, str) else expr

        elif 'output_template' in step:
            ctx['output'] = interpolate(step['output_template'], ctx)

    return ctx

# ════════════════════════════════════════════════════════════════════════════
# WORLD CONTEXT
# ════════════════════════════════════════════════════════════════════════════

class WorldContext:
    """
    Encapsulates all world state. Create one instance per session.
    Seed is displayed in every adventure so sessions are reproducible.

    Usage:
        world = WorldContext('mnt/data/world_mod')          # random seed
        world = WorldContext('mnt/data/world_mod', seed=42) # fixed seed
        adventure = world.generate_one_shot(party_level=3)
        world.save(adventure)
    """

    def __init__(self, world_dir: str, seed: Optional[int] = None):
        global CURRENT_ROOT, CURRENT_RNG, _TABLE_CACHE
        self.world_dir = world_dir
        self.seed = seed if seed is not None else int(time.time()) % 999983
        self.root = load_world_folder(world_dir)
        self.rng  = make_rng(self.seed)
        build_table_cache(self.root)
        CURRENT_ROOT = self.root
        CURRENT_RNG  = self.rng
        print(f"🌍 World loaded | seed: {self.seed} | tables: {len(_TABLE_CACHE)}")
        gens = list(deep_get(self.root, PROCGEN_GENERATORS).keys())
        print(f"   generators: {gens}")

    # ── Low-level helpers ────────────────────────────────────────────────────

    def roll_table(self, table_id: str) -> Any:
        return roll_table(get_proc_table(self.root, table_id), self.rng)

    def generate(self, gen_name: str, base_ctx: Optional[Dict] = None) -> Dict:
        return run_generator(self.root, gen_name, self.rng, base_ctx)

    # ── One-shot adventure generator ─────────────────────────────────────────

    def generate_one_shot(self, party_level: int = 3) -> Dict:
        """Generate a complete one-shot adventure for a party of a given level."""
        cr_guess = max(1, party_level - 1)
        ctx = self.generate("one_shot", base_ctx={"cr_guess": cr_guess, "party_level": party_level})
        rumor2 = self.roll_table("rumor")
        clue2  = self.roll_table("mystery_clue")
        region_id = (ctx.get("region") or {}).get("id", "") if isinstance(ctx.get("region"), dict) else ""
        live_events = self._get_live_events(region_id)
        return self._assemble(ctx, party_level, rumor2, clue2, live_events)

    # ── Session zero pitch ───────────────────────────────────────────────────

    def generate_session_zero(self) -> Dict:
        """Generate a brief player-facing pitch for session zero."""
        ctx = self.generate("session_zero")
        return {"seed": self.seed, "session_zero": ctx}

    # ── Save helper ──────────────────────────────────────────────────────────

    def save(self, adventure: Dict, prefix: str = "adventure", export_dir: str = "mnt/data/exports") -> str:
        """Render adventure to Markdown and save to disk."""
        md = render_markdown(adventure)
        os.makedirs(export_dir, exist_ok=True)
        ts   = datetime.now().strftime("%Y%m%d-%H%M%S")
        path = os.path.join(export_dir, f"{prefix}-{self.seed}-{ts}.md")
        with open(path, "w", encoding="utf-8") as f:
            f.write(md)
        print(f"✅ Saved: {path}")
        return path

    # ── Internal assembly ────────────────────────────────────────────────────

    def _get_live_events(self, region_id: str, count: int = 6) -> List[str]:
        all_events = (self.root.get("encounters") or {}).get("live_events") or {}
        pool = list(all_events.get(region_id) or [])
        if not pool:
            pool = [e for events in all_events.values() for e in events]
        self.rng.shuffle(pool)
        return pool[:count]

    def _str(self, val: Any, fallback: str = "") -> str:
        return str(val) if val is not None else fallback

    def _dict_get(self, d: Any, key: str, fallback: str = "") -> str:
        return str(d.get(key, fallback)) if isinstance(d, dict) else fallback

    def _assemble(self, ctx, party_level, rumor2, clue2, live_events) -> Dict:
        region         = ctx.get("region") or {}
        region_label   = self._dict_get(region, "label", "Unknown Region")
        region_id      = self._dict_get(region, "id", "")

        hook           = ctx.get("hook_types") or {}
        hook_type      = self._dict_get(hook, "type", "adventure").replace("_", " ").title()
        hook_twist     = self._dict_get(hook, "twist", "")

        dungeon        = ctx.get("dungeon_theme") or {}
        dungeon_name   = self._dict_get(dungeon, "name", "Ancient Site")
        dungeon_tags   = dungeon.get("tags", []) if isinstance(dungeon, dict) else []

        stype          = ctx.get("settlement_type") or {}
        stype_label    = self._dict_get(stype, "label", "settlement")
        sprob          = self._str(ctx.get("settlement_problem"), "none obvious yet")

        npc            = ctx.get("npc_personality") or {}
        npc_arch       = self._dict_get(npc, "archetype", "stranger").replace("_", " ").title()
        npc_motiv      = self._dict_get(npc, "motivation", "")
        npc_speech     = self._dict_get(npc, "speech_quirk", "")
        npc_secret     = self._dict_get(npc, "secret", "")

        rumor1         = ctx.get("rumor") or {}
        r1_text        = self._dict_get(rumor1, "text", "")
        r1_true        = rumor1.get("accurate", False) if isinstance(rumor1, dict) else False
        r2_text        = self._dict_get(rumor2, "text", "") if isinstance(rumor2, dict) else self._str(rumor2)
        r2_true        = rumor2.get("accurate", False) if isinstance(rumor2, dict) else False

        clue1          = ctx.get("mystery_clue") or {}
        c1_text        = self._dict_get(clue1, "clue", "")
        c1_impl        = self._dict_get(clue1, "implication", "")
        c1_type        = self._dict_get(clue1, "type", "")
        c2_text        = self._dict_get(clue2, "clue", "") if isinstance(clue2, dict) else self._str(clue2)
        c2_impl        = self._dict_get(clue2, "implication", "") if isinstance(clue2, dict) else ""
        c2_type        = self._dict_get(clue2, "type", "") if isinstance(clue2, dict) else ""

        weather        = ctx.get("weather_event") or {}
        wx_name        = self._dict_get(weather, "name", "")
        wx_desc        = self._dict_get(weather, "description", "")
        wx_effect      = self._dict_get(weather, "mechanical_effect", "")

        enc            = ctx.get("encounter_mix") or {}
        enc_type       = self._dict_get(enc, "type", "").replace("_", " ").title()
        enc_note       = self._dict_get(enc, "note", "")
        enc_combat     = enc.get("combat", 2) if isinstance(enc, dict) else 2
        enc_social     = enc.get("social", 1) if isinstance(enc, dict) else 1
        enc_puzzle     = enc.get("puzzle", 1) if isinstance(enc, dict) else 1
        enc_trap       = enc.get("trap", 1) if isinstance(enc, dict) else 1

        quirk          = ctx.get("magic_item_quirk") or {}
        q_text         = self._dict_get(quirk, "quirk", "")
        q_effect       = self._dict_get(quirk, "effect", "")

        complication   = self._str(ctx.get("complication"), "")
        reward         = self._str(ctx.get("reward"), "standard")
        cr_guess       = ctx.get("cr_guess", max(1, party_level - 1))

        # Pull bestiary monsters for this region
        bestiary = self.root.get("bestiary") or []
        regional = [b for b in bestiary if region_id in (b.get("habitats") or [])]
        if not regional:
            regional = [b for b in bestiary if abs(float(b.get("cr", 0)) - cr_guess) <= 2]
        if not regional:
            regional = bestiary[:4]
        sampled = self.rng.sample(regional, min(3, len(regional)))
        monster_encounters = [
            {
                "name":  b.get("id", "unknown").replace("_", " ").title(),
                "type":  ", ".join(b.get("tags") or []),
                "cr":    b.get("cr", "?"),
                "notes": f"Habitats: {', '.join(b.get('habitats') or [])}",
            }
            for b in sampled
        ]

        # Pull spawn table for random encounters
        spawn_tables = (self.root.get("encounters") or {}).get("spawn_tables") or []
        region_spawn = next((s for s in spawn_tables if s.get("region") == region_id), None)
        random_encounters = []
        if region_spawn:
            for i, e in enumerate(region_spawn.get("entries", [])[:8], 1):
                random_encounters.append({
                    "roll":      str(i),
                    "encounter": f"{e.get('creature', '?').replace('_', ' ').title()} (CR {e.get('cr', '?')})",
                })

        # Treasure from loot tables
        loot    = self.root.get("loot") or {}
        bundles = loot.get("bundles") or {}
        relics  = loot.get("cataclysm_relics") or {}
        bundle  = bundles.get(reward) or bundles.get("standard") or {}
        minor_relics = relics.get("minor") or ["lumenshard"]
        major_relics = relics.get("major") or ["hourglass_of_ages"]
        treasure = [
            {"roll": "1-2", "treasure": f"{bundle.get('dice', '2d10')} {bundle.get('unit', 'gold')}"},
            {"roll": "3-4", "treasure": f"Minor relic: {self.rng.choice(minor_relics).replace('_', ' ')}"},
            {"roll": "5-6", "treasure": f"Major relic: {self.rng.choice(major_relics).replace('_', ' ')}"},
        ]

        # Faction
        factions = self.root.get("factions") or []
        faction  = next((f for f in factions if f.get("base_region") == region_id), None)
        if not faction and factions:
            faction = self.rng.choice(factions)
        faction_name  = (faction or {}).get("name", "")
        faction_goals = (faction or {}).get("goals") or []

        # Known NPC from this region
        npcs_yaml = self.root.get("npcs") or []
        reg_npcs  = [n for n in npcs_yaml if n.get("location") == region_id] or npcs_yaml
        known_npc = self.rng.choice(reg_npcs) if reg_npcs else {}

        title = f"{hook_type} in {region_label}"

        return {
            "seed":        self.seed,
            "party_level": party_level,
            "adventure": {
                "title":  title,
                "region": region_label,
                "theme":  hook_type.lower().replace(" ", "_"),
                "quest_summary": {
                    "type":         hook_type,
                    "twist":        hook_twist,
                    "complication": complication,
                    "reward":       reward,
                    "cr_range":     f"CR {max(1, cr_guess - 1)}-{cr_guess + 2}",
                },
                "encounter_structure": {
                    "type":   enc_type,
                    "note":   enc_note,
                    "combat": enc_combat,
                    "social": enc_social,
                    "puzzle": enc_puzzle,
                    "trap":   enc_trap,
                },
                "hooks": [
                    {"title": "Quest Giver",  "description": f"A {npc_arch} needs help urgently: {npc_motiv}"},
                    {"title": "Tavern Rumor", "description": r1_text},
                    {"title": "Strange Clue", "description": c1_text},
                ],
                "quest_giver": {
                    "archetype":        npc_arch,
                    "motivation":       npc_motiv,
                    "speech_quirk":     npc_speech,
                    "secret":           npc_secret,
                    "known_npc":        known_npc.get("name", ""),
                    "known_npc_traits": ", ".join(known_npc.get("traits") or []),
                },
                "locations": {
                    "settlement": {
                        "name":            f"{region_label} {stype_label.title()}",
                        "type":            stype_label,
                        "current_problem": sprob,
                    },
                    "dungeon": {
                        "name":     dungeon_name,
                        "themes":   ", ".join(str(t) for t in dungeon_tags),
                        "depth":    self.rng.randint(2, 5),
                        "cr_guess": cr_guess,
                    },
                },
                "weather": {
                    "name":              wx_name,
                    "description":       wx_desc,
                    "mechanical_effect": wx_effect,
                },
                "rumors": [
                    {"text": r1_text, "true": r1_true},
                    {"text": r2_text, "true": r2_true},
                ],
                "clues": [
                    {"type": c1_type, "clue": c1_text, "implication": c1_impl},
                    {"type": c2_type, "clue": c2_text, "implication": c2_impl},
                ],
                "magic_item_quirk": {"quirk": q_text, "effect": q_effect},
                "monster_encounters": monster_encounters,
                "random_encounters":  random_encounters,
                "treasure":           treasure,
                "live_events":        live_events,
                "faction": {
                    "name":  faction_name,
                    "goals": ", ".join(str(g) for g in faction_goals),
                },
                "resolution": {
                    "choices": [
                        f"Side with {faction_name} to resolve the crisis." if faction_name else "Confront the threat directly.",
                        "Exploit the situation for personal gain.",
                        "Forge an unexpected alliance with the antagonist.",
                    ],
                    "outcome": f"The fate of {region_label} shifts based on the party's choices.",
                },
            },
        }

In [2]:
# ════════════════════════════════════════════════════════════════════════════
# MARKDOWN EXPORTER
# ════════════════════════════════════════════════════════════════════════════

def _g(d, *keys, fallback=""):
    """Safe nested dict getter."""
    cur = d
    for k in keys:
        if isinstance(cur, dict):
            cur = cur.get(k, fallback)
        else:
            return fallback
    return cur if cur is not None else fallback

def _table(rows, headers):
    """Render a list of dicts as a Markdown table."""
    if not rows:
        return ""
    sep  = "|" + "|".join(["---"] * len(headers)) + "|"
    head = "|" + "|".join(h.replace("_", " ").title() for h in headers) + "|"
    body = "\n".join(
        "|" + "|".join(str(r.get(h, "")) for h in headers) + "|"
        for r in rows
    )
    return "\n".join([head, sep, body])


def render_markdown(data: dict) -> str:
    """
    Render a WorldContext adventure dict to game-ready Markdown.
    The 'seed' field at the top enables reproducibility.
    """
    seed        = data.get("seed", "?")
    party_level = data.get("party_level", "?")
    adv         = data.get("adventure") or data

    title  = _g(adv, "title",  fallback="Untitled Adventure")
    region = _g(adv, "region", fallback="Unknown")

    qs         = adv.get("quest_summary") or {}
    q_type     = _g(qs, "type")
    q_twist    = _g(qs, "twist")
    q_comp     = _g(qs, "complication")
    q_reward   = _g(qs, "reward")
    q_cr       = _g(qs, "cr_range")

    enc        = adv.get("encounter_structure") or {}
    hooks      = adv.get("hooks") or []
    qg         = adv.get("quest_giver") or {}
    locs       = adv.get("locations") or {}
    settlement = locs.get("settlement") or {}
    dungeon    = locs.get("dungeon") or {}
    weather    = adv.get("weather") or {}
    rumors     = adv.get("rumors") or []
    clues      = adv.get("clues") or []
    quirk      = adv.get("magic_item_quirk") or {}
    monsters   = adv.get("monster_encounters") or []
    randoms    = adv.get("random_encounters") or []
    treasure   = adv.get("treasure") or []
    live_ev    = adv.get("live_events") or []
    faction    = adv.get("faction") or {}
    resolution = adv.get("resolution") or {}

    lines = []
    A = lines.append

    # ── Header ───────────────────────────────────────────────────────────────
    A(f"# 🗺️ {title}")
    A(f"> 🎲 **Seed:** `{seed}` | **Party Level:** {party_level}")
    A("> *Regenerate: `world = WorldContext(WORLD_DIR, seed={seed})`*".format(seed=seed))
    A("")

    # ── Quest summary ────────────────────────────────────────────────────────
    A("## 📜 Quest Summary")
    A(f"| | |")
    A(f"|---|---|")
    A(f"| **Region** | {region} |")
    if q_type:   A(f"| **Type** | {q_type} |")
    if q_twist:  A(f"| **Twist** | {q_twist} |")
    if q_comp:   A(f"| **Complication** | {q_comp} |")
    if q_reward: A(f"| **Reward Tier** | {q_reward} |")
    if q_cr:     A(f"| **CR Range** | {q_cr} |")
    A("")

    # ── Encounter structure ──────────────────────────────────────────────────
    if enc:
        A("## ⚔️ Encounter Structure")
        enc_type = _g(enc, "type")
        enc_note = _g(enc, "note")
        if enc_type: A(f"**Style:** {enc_type}  ")
        if enc_note: A(f"*{enc_note}*  ")
        parts = []
        for k in ["combat", "social", "puzzle", "trap"]:
            v = enc.get(k)
            if v:
                parts.append(f"{v}× {k}")
        if parts:
            A("**Recommended mix:** " + ", ".join(parts))
        A("")

    # ── Adventure hooks ──────────────────────────────────────────────────────
    if hooks:
        A("## 🎣 Adventure Hooks")
        for i, h in enumerate(hooks, 1):
            t = h.get("title", f"Hook {i}")
            d = h.get("description", "")
            A(f"{i}. **{t}** — {d}")
        A("")

    # ── Quest giver ──────────────────────────────────────────────────────────
    if qg:
        A("## 🧑 Quest Giver")
        if _g(qg, "known_npc"): A(f"**Known NPC:** {_g(qg, 'known_npc')} ({_g(qg, 'known_npc_traits')})  ")
        if _g(qg, "archetype"): A(f"**Archetype:** {_g(qg, 'archetype')}  ")
        if _g(qg, "motivation"): A(f"**Motivation:** {_g(qg, 'motivation')}  ")
        if _g(qg, "speech_quirk"): A(f"**Speech Quirk:** {_g(qg, 'speech_quirk')}  ")
        if _g(qg, "secret"): A(f"**Secret (GM only):** _{_g(qg, 'secret')}_  ")
        A("")

    # ── Settlement ───────────────────────────────────────────────────────────
    if settlement:
        A("## 🏙️ Settlement")
        if _g(settlement, "name"): A(f"**Name:** {_g(settlement, 'name')}  ")
        if _g(settlement, "type"): A(f"**Type:** {_g(settlement, 'type')}  ")
        if _g(settlement, "current_problem"): A(f"**Current Problem:** {_g(settlement, 'current_problem')}  ")
        A("")

    # ── Dungeon ──────────────────────────────────────────────────────────────
    if dungeon:
        A("## 🏚️ Dungeon")
        if _g(dungeon, "name"): A(f"**Name:** {_g(dungeon, 'name')}  ")
        if _g(dungeon, "themes"): A(f"**Themes:** {_g(dungeon, 'themes')}  ")
        depth    = dungeon.get("depth")
        cr_guess = dungeon.get("cr_guess")
        if depth:    A(f"**Depth:** {depth} levels  ")
        if cr_guess: A(f"**CR Baseline:** {cr_guess}  ")
        A("")

    # ── Weather ──────────────────────────────────────────────────────────────
    if weather and _g(weather, "name"):
        A("## 🌩️ Weather Condition")
        A(f"**{_g(weather, 'name')}** — {_g(weather, 'description')}  ")
        if _g(weather, "mechanical_effect"):
            A(f"*Effect:* {_g(weather, 'mechanical_effect')}")
        A("")

    # ── Rumors ───────────────────────────────────────────────────────────────
    if rumors:
        A("## 💬 Rumors at the Tavern")
        A("*Share with players. Items marked ✓ are true; items marked ✗ are false.*")
        A("")
        for r in rumors:
            mark = "✓" if r.get("true") else "✗"
            A(f"- [{mark}] {r.get('text', '')}")
        A("")

    # ── Clues ────────────────────────────────────────────────────────────────
    if clues:
        A("## 🔍 Investigation Clues")
        for c in clues:
            ctype = c.get("type", "").title()
            ctext = c.get("clue", "")
            impl  = c.get("implication", "")
            if ctext:
                A(f"**[{ctype}]** {ctext}")
                if impl:
                    A(f"   → *{impl}*")
        A("")

    # ── Faction ──────────────────────────────────────────────────────────────
    if faction and _g(faction, "name"):
        A("## 🏛️ Active Faction")
        A(f"**{_g(faction, 'name')}**  ")
        if _g(faction, "goals"): A(f"*Goals:* {_g(faction, 'goals')}")
        A("")

    # ── Monster encounters ───────────────────────────────────────────────────
    if monsters:
        A("## 👹 Key Monsters")
        A(_table(monsters, ["name", "type", "cr", "notes"]))
        A("")

    # ── Random encounter table ───────────────────────────────────────────────
    if randoms:
        A("## 🎲 Random Encounter Table")
        A(_table(randoms, ["roll", "encounter"]))
        A("")

    # ── Treasure ─────────────────────────────────────────────────────────────
    if treasure:
        A("## 💰 Treasure")
        A(_table(treasure, ["roll", "treasure"]))
        A("")

    # ── Magic item quirk ─────────────────────────────────────────────────────
    if quirk and _g(quirk, "quirk"):
        A("## ✨ Magic Item Quirk")
        A(f"**{_g(quirk, 'quirk')}** {_g(quirk, 'effect')}")
        A("")

    # ── Live events ──────────────────────────────────────────────────────────
    if live_ev:
        A("## ⚡ Live Events (Roll d6 Mid-Session)")
        A("*Drop one of these into a slow moment to inject energy.*")
        A("")
        for i, ev in enumerate(live_ev, 1):
            text = ev if isinstance(ev, str) else ev.get("event", str(ev))
            A(f"{i}. {text}")
        A("")

    # ── Resolution ───────────────────────────────────────────────────────────
    if resolution:
        A("## 🎯 Resolution Paths")
        for c in resolution.get("choices") or []:
            A(f"- {c}")
        outcome = resolution.get("outcome", "")
        if outcome:
            A(f"\n**Outcome:** {outcome}")
        A("")

    # ── Footer ───────────────────────────────────────────────────────────────
    A("---")
    A(f"*Generated by Etheras World Generator v12 | Seed `{seed}` | {datetime.now().strftime('%Y-%m-%d')}*")

    return "\n".join(lines)

In [3]:
# ════════════════════════════════════════════════════════════════════════════
# WORKFLOW — run this cell to generate a complete one-shot adventure
# ════════════════════════════════════════════════════════════════════════════

WORLD_DIR   = os.environ.get("WORLD_DIR", "mnt/data/world_mod")
PARTY_LEVEL = 3   # change this to match your party
SEED        = None  # set to an int for a reproducible adventure, e.g. SEED = 42

world     = WorldContext(WORLD_DIR, seed=SEED)
adventure = world.generate_one_shot(party_level=PARTY_LEVEL)
path      = world.save(adventure)

print(f"\n{'='*60}")
print(f"Adventure: {adventure['adventure']['title']}")
print(f"Seed:      {adventure['seed']}")
print(f"File:      {path}")
print(f"{'='*60}")
print("\nDrop this file into Claude Desktop for full narrative expansion.")
print("To regenerate the exact same adventure: SEED =", adventure['seed'])

🌍 World loaded | seed: 17602 | tables: 12
   generators: ['settlement', 'dungeon', 'quest', 'one_shot', 'session_zero']
✅ Saved: mnt/data/exports/adventure-17602-20260528-125542.md

Adventure: Dungeon Delving in Glitterhold Forest
Seed:      17602
File:      mnt/data/exports/adventure-17602-20260528-125542.md

Drop this file into Claude Desktop for full narrative expansion.
To regenerate the exact same adventure: SEED = 17602


/var/folders/0w/zzlz_4p5729f8vc_5_17kmg00000gq/T/ipykernel_54651/1610016469.py:251: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ctx['now'] = datetime.now(datetime.timezone.utc).isoformat() if hasattr(datetime, 'timezone') else datetime.utcnow().isoformat()


In [4]:
# ════════════════════════════════════════════════════════════════════════════
# OPTIONAL: Session Zero Pitch
# Generates a brief player-facing premise. Share before session zero.
# ════════════════════════════════════════════════════════════════════════════

sz = world.generate_session_zero()
print(sz["session_zero"].get("output", "(no output template)"))

# Session Zero Pitch
**Region:** Glitterhold Forest
**Quest Hook:** Diplomacy — Both sides are secretly already at war.
**Location:** Haunted Noble House (['haunted', 'noble', 'relic_cache'])
**Quest Giver Archetype:** wandering_herbalist — Heal and move on.
**Opening Rumor:** The old healer refuses to treat Cataclysm wounds. Says she's seen what comes after.



/var/folders/0w/zzlz_4p5729f8vc_5_17kmg00000gq/T/ipykernel_54651/1610016469.py:251: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ctx['now'] = datetime.now(datetime.timezone.utc).isoformat() if hasattr(datetime, 'timezone') else datetime.utcnow().isoformat()


In [5]:
# ════════════════════════════════════════════════════════════════════════════
# OPTIONAL: Print raw adventure dict (for debugging)
# ════════════════════════════════════════════════════════════════════════════

from pprint import pprint
pprint(adventure)

{'adventure': {'clues': [{'clue': 'The survivor refuses to look anyone in the '
                                  'eye since returning from the ruins.',
                          'implication': 'Saw something that is still '
                                         'watching.',
                          'type': 'testimonial'},
                         {'clue': 'The child says the bad man smelled like '
                                  "rain and smoke. It hasn't rained in a "
                                  'month.',
                          'implication': 'Cataclysm echo or dimensional '
                                         'traveler.',
                          'type': 'testimonial'}],
               'encounter_structure': {'combat': 3,
                                       'note': 'Good for dungeon delving and '
                                               'monster hunts.',
                                       'puzzle': 1,
                                       'social':

In [6]:
# ════════════════════════════════════════════════════════════════════════════
# OPTIONAL: Print rendered Markdown in notebook
# ════════════════════════════════════════════════════════════════════════════

from IPython.display import Markdown, display
display(Markdown(render_markdown(adventure)))

# 🗺️ Dungeon Delving in Glitterhold Forest
> 🎲 **Seed:** `17602` | **Party Level:** 3
> *Regenerate: `world = WorldContext(WORLD_DIR, seed=17602)`*

## 📜 Quest Summary
| | |
|---|---|
| **Region** | Glitterhold Forest |
| **Type** | Dungeon Delving |
| **Twist** | Time flows strangely within the ruin. |
| **Complication** | Supplies are running low faster than expected. |
| **Reward Tier** | poor |
| **CR Range** | CR 1-4 |

## ⚔️ Encounter Structure
**Style:** Combat Heavy  
*Good for dungeon delving and monster hunts.*  
**Recommended mix:** 3× combat, 1× social, 1× puzzle, 1× trap

## 🎣 Adventure Hooks
1. **Quest Giver** — A Obsessed Collector needs help urgently: Complete the collection at any price.
2. **Tavern Rumor** — The lord's personal guard has been doubled. Something spooked him.
3. **Strange Clue** — The survivor refuses to look anyone in the eye since returning from the ruins.

## 🧑 Quest Giver
**Known NPC:** Selthir Moonbough (quiet, precise, reverent)  
**Archetype:** Obsessed Collector  
**Motivation:** Complete the collection at any price.  
**Speech Quirk:** Catalogs everything they see mentally — sometimes aloud.  
**Secret (GM only):** _The collection is cursed._  

## 🏙️ Settlement
**Name:** Glitterhold Forest Town  
**Type:** town  
**Current Problem:** Children going missing near the old mill.  

## 🏚️ Dungeon
**Name:** Cataclysmic Ruins  
**Themes:** temporal, unstable, relic_rich  
**Depth:** 3 levels  
**CR Baseline:** 2  

## 🌩️ Weather Condition
**Witchlight** — Green fire burns wherever light already burns.  
*Effect:* Conjuration/divination at advantage; fire spells cast at +1 level.

## 💬 Rumors at the Tavern
*Share with players. Items marked ✓ are true; items marked ✗ are false.*

- [✓] The lord's personal guard has been doubled. Something spooked him.
- [✓] The forest has been completely quiet for a week. Not even birds.

## 🔍 Investigation Clues
**[Testimonial]** The survivor refuses to look anyone in the eye since returning from the ruins.
   → *Saw something that is still watching.*
**[Testimonial]** The child says the bad man smelled like rain and smoke. It hasn't rained in a month.
   → *Cataclysm echo or dimensional traveler.*

## 🏛️ Active Faction
**Concord of the Wild Oath**  

## 👹 Key Monsters
|Name|Type|Cr|Notes|
|---|---|---|---|
|Troll|regenerating, brute|5|Habitats: glitterhold_forest, deep_swamps, caves|

## 🎲 Random Encounter Table
|Roll|Encounter|
|---|---|
|1|Wolf Pack (CR 1)|
|2|Troll (CR 5)|
|3|Ghoul (CR 2)|
|4|Dryad (CR 2)|

## 💰 Treasure
|Roll|Treasure|
|---|---|
|1-2|2d10 copper|
|3-4|Minor relic: lumenshard|
|5-6|Major relic: tome of unmaking|

## ✨ Magic Item Quirk
**Hates fire.** Disadvantage on fire attack rolls; advantage on saves against fire.

## ⚡ Live Events (Roll d6 Mid-Session)
*Drop one of these into a slow moment to inject energy.*

1. A pyre of wildflowers arranged in a spiral — still warm, no tracks around it.
2. A moss-covered door stands between two ancient trees. A light is visible beneath the door.
3. A path that wasn't there yesterday leads to a clearing not on any map.
4. The canopy parts overhead to show a perfect circle of sky — no stars visible.
5. Selthir Moonbough passes the party without acknowledgment — looking hunted.
6. A wolf pack flanks the party for an hour, maintaining perfect distance.

## 🎯 Resolution Paths
- Side with Concord of the Wild Oath to resolve the crisis.
- Exploit the situation for personal gain.
- Forge an unexpected alliance with the antagonist.

**Outcome:** The fate of Glitterhold Forest shifts based on the party's choices.

---
*Generated by Etheras World Generator v12 | Seed `17602` | 2026-05-28*